# 5장. 분석을 믿을 수 있게 만드는 데이터 전처리

이 노트북은 `book/chapters/ch05_data_preprocessing.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 데이터를 무조건 깨끗하게 만드는 것이 아니라, 분석 결과를 신뢰할 수 있도록 데이터의 상태를 확인하고 처리 기준을 남기는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 원본 데이터는 `data/raw/`에 그대로 둡니다.
- 전처리 결과는 `data/processed/`에 별도로 저장합니다.
- 전처리 요약 보고서는 `reports/ch05_preprocessing_summary.md`에 저장합니다.
- 코드가 실행되었다고 끝이 아니라, 처리 기준을 설명할 수 있어야 합니다.


## 1. 왜 전처리가 필요한가

현실의 데이터는 처음부터 분석하기 좋은 형태로 주어지지 않습니다.

예를 들어 다음과 같은 문제가 있을 수 있습니다.

| 문제 유형 | 예시 | 확인할 질문 |
|---|---|---|
| 결측치 | 나이가 비어 있는 고객 | 비어 있는 이유가 무엇인가? 삭제해도 되는가? |
| 중복 | 같은 고객 ID가 여러 번 등장 | 자연스러운 중복인가, 데이터 오류인가? |
| 타입 오류 | 가격이 문자열로 저장됨 | 계산 가능한 숫자형으로 바꿀 수 있는가? |
| 날짜 오류 | 주문일이 문자열로 저장됨 | 월별·요일별 분석에 사용할 수 있는가? |
| 문자열 표기 차이 | `Seoul`, ` Seoul`, `SEOUL` | 같은 값을 하나의 표기로 통일해야 하는가? |
| 이상값 | 수량이 음수이거나 가격이 0원 | 실제 의미가 있는 값인가, 입력 오류인가? |
| 파생 컬럼 필요 | 수량과 단가만 있고 주문금액이 없음 | 분석에 필요한 새 컬럼을 만들 수 있는가? |


## 2. 원본 데이터와 전처리 데이터 분리하기

전처리에서 가장 중요한 원칙은 원본 데이터를 직접 덮어쓰지 않는 것입니다.

```text
data/
├─ raw/
│  ├─ customers.csv
│  ├─ products.csv
│  ├─ orders.csv
│  └─ order_items.csv
└─ processed/
   ├─ customers_clean.csv
   ├─ products_clean.csv
   ├─ orders_clean.csv
   └─ order_items_clean.csv
```

원본과 전처리 결과를 분리하면 실수했을 때 원본으로 돌아갈 수 있고, 전처리 전후 차이를 비교할 수 있습니다.


## 3. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [37]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('원본 데이터 폴더:', RAW_DIR)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: c:\dev\llm-data-analysis-study\chapter05
프로젝트 루트: c:\dev\llm-data-analysis-study\chapter05
원본 데이터 폴더: ..\data\raw
전처리 데이터 폴더: c:\dev\llm-data-analysis-study\chapter05\data\processed
보고서 폴더: c:\dev\llm-data-analysis-study\chapter05\reports


## 4. 원본 데이터 불러오기

이번 장에서는 온라인 쇼핑몰 예제 데이터 4개를 사용합니다.

- `customers.csv`: 고객 정보
- `products.csv`: 상품 정보
- `orders.csv`: 주문 정보
- `order_items.csv`: 주문 상세 정보


In [38]:
customers = pd.read_csv(RAW_DIR / 'customers.csv')
products = pd.read_csv(RAW_DIR / 'products.csv')
orders = pd.read_csv(RAW_DIR / 'orders.csv')
order_items = pd.read_csv(RAW_DIR / 'order_items.csv')

print('원본 데이터 불러오기 완료')


원본 데이터 불러오기 완료


## 5. 전처리 전 데이터 크기 기록하기

전처리 전 데이터 크기를 기록해 두면 나중에 어떤 데이터가 얼마나 바뀌었는지 비교할 수 있습니다.


In [39]:
raw_data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

raw_shapes = pd.DataFrame({
    'dataset': list(raw_data.keys()),
    'rows': [df.shape[0] for df in raw_data.values()],
    'columns': [df.shape[1] for df in raw_data.values()],
})

raw_shapes


,dataset,rows,columns
0,customers,150,6
1,products,100,4
2,orders,300,5
3,order_items,764,5


In [40]:
for name, df in raw_data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (100, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 5)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
shape: (764, 5)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



## 6. 원본을 복사해서 전처리 시작하기

전처리 과정에서는 원본 DataFrame을 직접 수정하지 않고 복사본을 사용합니다.


In [41]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

clean_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

print('전처리용 복사본 생성 완료')


전처리용 복사본 생성 완료


## 7. 결측치 확인하기

결측치를 처리하기 전에 먼저 어디에 얼마나 비어 있는 값이 있는지 확인합니다. 개수와 비율을 함께 보면 데이터 크기에 따른 차이를 이해하기 쉽습니다.


In [42]:
for name, df in clean_data.items():
    print(f'\n[{name}] 결측치 개수')
    print(df.isna().sum())



[customers] 결측치 개수
customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64

[products] 결측치 개수
product_id      0
product_name    0
category        0
price           0
dtype: int64

[orders] 결측치 개수
order_id          0
customer_id       0
order_date        0
payment_method    0
order_status      0
dtype: int64

[order_items] 결측치 개수
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
dtype: int64


In [43]:
def missing_summary(df):
    summary = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_ratio': (df.isna().mean() * 100).round(2),
    })
    return summary.sort_values('missing_count', ascending=False)

missing_summary(customers_clean)


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


## 8. 결측치 처리하기

결측치 처리는 컬럼의 의미에 따라 달라집니다.

- 나이처럼 숫자형 컬럼은 중앙값으로 대체할 수 있습니다.
- 도시처럼 범주형 컬럼은 `Unknown`으로 남겨 분석에서 구분할 수 있습니다.

실제 업무에서는 결측치가 왜 생겼는지 먼저 확인해야 합니다.


In [44]:
if 'age' in customers_clean.columns:
    customers_clean['age'] = pd.to_numeric(customers_clean['age'], errors='coerce')
    age_median = customers_clean['age'].median()
    customers_clean['age'] = customers_clean['age'].fillna(age_median)
    print('age 중앙값:', age_median)

if 'city' in customers_clean.columns:
    customers_clean['city'] = customers_clean['city'].fillna('Unknown')

missing_summary(customers_clean)


age 중앙값: 40.0


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


## 9. 중복 확인하기

중복은 맥락에 따라 의미가 다릅니다. `customers`의 `customer_id` 중복은 오류일 가능성이 크지만, `order_items`의 `order_id` 반복은 한 주문에 여러 상품이 들어갈 수 있으므로 자연스럽습니다.


In [45]:
for name, df in clean_data.items():
    print(name, '전체 행 중복 수:', df.duplicated().sum())


customers 전체 행 중복 수: 0
products 전체 행 중복 수: 0
orders 전체 행 중복 수: 0
order_items 전체 행 중복 수: 0


In [46]:
key_checks = {
    'customers': ('customer_id', customers_clean),
    'products': ('product_id', products_clean),
    'orders': ('order_id', orders_clean),
    'order_items': ('order_item_id', order_items_clean),
}

for name, (key_col, df) in key_checks.items():
    if key_col in df.columns:
        print(name, key_col, '중복 수:', df[key_col].duplicated().sum())
    else:
        print(name, key_col, '컬럼 없음')


customers customer_id 중복 수: 0
products product_id 중복 수: 0
orders order_id 중복 수: 0
order_items order_item_id 중복 수: 0


In [47]:
customers_clean = customers_clean.drop_duplicates()
products_clean = products_clean.drop_duplicates()
orders_clean = orders_clean.drop_duplicates()
order_items_clean = order_items_clean.drop_duplicates()

print('완전 중복 행 제거 완료')


완전 중복 행 제거 완료


## 10. 문자열 표기 정리하기

문자열 데이터에는 앞뒤 공백이 숨어 있을 수 있습니다. `Seoul`과 ` Seoul `은 눈으로는 비슷하지만 pandas에서는 서로 다른 값입니다.

결측치를 문자열 `nan`으로 바꾸지 않도록 주의하면서 문자열 컬럼의 앞뒤 공백을 제거합니다.


In [48]:
def strip_string_columns(df):
    result = df.copy()
    string_columns = result.select_dtypes(include='object').columns

    for col in string_columns:
        result[col] = result[col].where(
            result[col].isna(),
            result[col].astype(str).str.strip()
        )

    return result

customers_clean = strip_string_columns(customers_clean)
products_clean = strip_string_columns(products_clean)
orders_clean = strip_string_columns(orders_clean)
order_items_clean = strip_string_columns(order_items_clean)

print('문자열 앞뒤 공백 제거 완료')


문자열 앞뒤 공백 제거 완료


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_1904\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_1904\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pan

In [49]:
if 'city' in customers_clean.columns:
    display(customers_clean['city'].value_counts().head(10))

if 'order_status' in orders_clean.columns:
    display(orders_clean['order_status'].value_counts())


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 11. 주문 상태값 통일하기

주문 상태값이 `complete`, `Complete`, `COMPLETED`, `완료`처럼 섞여 있으면 같은 상태를 서로 다른 값으로 집계할 수 있습니다. 대표 표기로 통일합니다.


In [50]:
status_map = {
    'complete': 'completed',
    'Complete': 'completed',
    'COMPLETED': 'completed',
    '완료': 'completed',
    'cancel': 'cancelled',
    'Cancel': 'cancelled',
    'CANCELLED': 'cancelled',
    '취소': 'cancelled',
    'refund': 'refunded',
    'Refund': 'refunded',
    'REFUNDED': 'refunded',
    '환불': 'refunded',
}

if 'order_status' in orders_clean.columns:
    orders_clean['order_status'] = orders_clean['order_status'].replace(status_map)
    display(orders_clean['order_status'].value_counts())


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 12. 날짜 컬럼 변환하기

날짜처럼 보이는 값도 실제로는 문자열일 수 있습니다. 월별 주문 분석이나 요일별 분석을 하려면 날짜형으로 변환해야 합니다.

`errors="coerce"`는 변환할 수 없는 값을 `NaT`로 바꿉니다. 그래서 변환 후 실패 건수를 반드시 확인해야 합니다.


In [51]:
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')
print('order_date 변환 실패:', orders_clean['order_date'].isna().sum())
print('주문 시작일:', orders_clean['order_date'].min())
print('주문 종료일:', orders_clean['order_date'].max())

orders_clean['order_month'] = orders_clean['order_date'].dt.to_period('M').astype(str)
orders_clean['order_dayofweek'] = orders_clean['order_date'].dt.day_name()

orders_clean[['order_date', 'order_month', 'order_dayofweek']].head()


order_date 변환 실패: 0
주문 시작일: 2025-09-18 00:00:00
주문 종료일: 2026-09-17 00:00:00


,order_date,order_month,order_dayofweek
0,2026-07-17,2026-07,Friday
1,2025-10-02,2025-10,Thursday
2,2026-01-29,2026-01,Thursday
3,2026-04-11,2026-04,Saturday
4,2026-03-02,2026-03,Monday


In [52]:
if 'signup_date' in customers_clean.columns:
    customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'], errors='coerce')
    print('signup_date 변환 실패:', customers_clean['signup_date'].isna().sum())
    display(customers_clean[['customer_id', 'signup_date']].head())


signup_date 변환 실패: 0


,customer_id,signup_date
0,1,2024-08-29
1,2,2026-01-12
2,3,2024-08-22
3,4,2026-06-23
4,5,2024-11-23


## 13. 숫자형 컬럼 변환하기

가격, 수량, 단가는 숫자처럼 보여도 문자열일 수 있습니다. 특히 `10,000`처럼 쉼표가 들어간 값은 바로 계산하기 어렵습니다.


In [53]:
def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

products_clean['price'] = to_number(products_clean['price'])
order_items_clean['quantity'] = to_number(order_items_clean['quantity'])
order_items_clean['unit_price'] = to_number(order_items_clean['unit_price'])

print('price 변환 실패:', products_clean['price'].isna().sum())
print('quantity 변환 실패:', order_items_clean['quantity'].isna().sum())
print('unit_price 변환 실패:', order_items_clean['unit_price'].isna().sum())


price 변환 실패: 0
quantity 변환 실패: 0
unit_price 변환 실패: 0


## 14. 이상값 후보 확인하기

이상값은 일반적인 범위를 벗어난 값입니다. 하지만 이상값이 항상 오류는 아닙니다. 0원 상품은 이벤트 상품일 수 있고, 음수 수량은 반품을 의미할 수도 있습니다. 먼저 확인하고 기준을 정해야 합니다.


In [54]:
display(products_clean['price'].describe())
display(order_items_clean[['quantity', 'unit_price']].describe())


count       100.000000
mean     110040.000000
std       56433.910574
min        5000.000000
25%       65750.000000
50%      112000.000000
75%      161000.000000
max      200000.000000
Name: price, dtype: float64

,quantity,unit_price
count,764.000000,764.000000
mean,3.053665,108561.518325
std,1.410873,56996.770604
min,1.000000,5000.000000
25%,2.000000,62000.000000
50%,3.000000,111000.000000
75%,4.000000,161250.000000
max,5.000000,200000.000000


In [55]:
print('price <= 0:', len(products_clean[products_clean['price'] <= 0]))
print('quantity <= 0:', len(order_items_clean[order_items_clean['quantity'] <= 0]))
print('unit_price <= 0:', len(order_items_clean[order_items_clean['unit_price'] <= 0]))


price <= 0: 0
quantity <= 0: 0
unit_price <= 0: 0


### 실습용 이상값 처리 기준

이번 실습에서는 정상 주문 분석을 위해 다음 기준을 적용합니다.

```text
- price가 0 이하인 상품은 분석 대상에서 제외
- quantity가 0 이하인 주문 상세는 분석 대상에서 제외
- unit_price가 0 이하인 주문 상세는 분석 대상에서 제외
```

실제 업무에서는 이 기준을 적용하기 전에 반드시 원본 시스템이나 담당자 확인이 필요합니다.


In [56]:
products_clean = products_clean[products_clean['price'] > 0]
order_items_clean = order_items_clean[order_items_clean['quantity'] > 0]
order_items_clean = order_items_clean[order_items_clean['unit_price'] > 0]

print('이상값 처리 후 products:', products_clean.shape)
print('이상값 처리 후 order_items:', order_items_clean.shape)


이상값 처리 후 products: (100, 4)
이상값 처리 후 order_items: (764, 5)


## 15. 파생 컬럼 만들기

전처리된 수량과 단가를 사용하면 주문 상세 금액을 계산할 수 있습니다.

`line_total = quantity × unit_price`


In [57]:
order_items_clean['line_total'] = (
    order_items_clean['quantity'] * order_items_clean['unit_price']
)

order_items_clean[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,3,102000,306000
1,5,25000,125000
2,3,142000,426000
3,3,193000,579000
4,4,189000,756000


In [58]:
print('전처리 후 주문 상세 금액 합계:', order_items_clean['line_total'].sum())


전처리 후 주문 상세 금액 합계: 255610000


## 16. 파일 간 관계 다시 확인하기

전처리 과정에서 일부 행을 삭제하면 파일 간 관계가 깨질 수 있습니다. 예를 들어 `products`에서 일부 상품을 제외했는데 `order_items`에는 그 상품 ID가 남아 있을 수 있습니다.


In [59]:
invalid_customers = orders_clean[
    ~orders_clean['customer_id'].isin(customers_clean['customer_id'])
]

invalid_orders = order_items_clean[
    ~order_items_clean['order_id'].isin(orders_clean['order_id'])
]

invalid_products = order_items_clean[
    ~order_items_clean['product_id'].isin(products_clean['product_id'])
]

relationship_checks = pd.DataFrame({
    'check': [
        'orders.customer_id exists in customers.customer_id',
        'order_items.order_id exists in orders.order_id',
        'order_items.product_id exists in products.product_id',
    ],
    'invalid_count': [
        len(invalid_customers),
        len(invalid_orders),
        len(invalid_products),
    ],
})

relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,0
1,order_items.order_id exists in orders.order_id,0
2,order_items.product_id exists in products.prod...,0


## 17. 전처리 전후 비교하기

전처리 전후 데이터 크기를 비교합니다. 행 수가 줄었다면 어떤 기준으로 줄었는지 설명할 수 있어야 하고, 열 수가 늘었다면 어떤 파생 컬럼이 추가되었는지 설명할 수 있어야 합니다.


In [60]:
processed_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

processed_shapes = pd.DataFrame({
    'dataset': list(processed_data.keys()),
    'rows': [df.shape[0] for df in processed_data.values()],
    'columns': [df.shape[1] for df in processed_data.values()],
})

comparison = raw_shapes.merge(
    processed_shapes,
    on='dataset',
    suffixes=('_raw', '_processed')
)

comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,150,6,150,6
1,products,100,4,100,4
2,orders,300,5,300,7
3,order_items,764,5,764,6


## 18. 전처리 결과 저장하기

전처리된 데이터는 `data/processed` 폴더에 저장합니다. Excel에서 한글 CSV를 바로 열 수 있도록 `utf-8-sig` 인코딩을 사용합니다.


In [61]:
customers_clean.to_csv(PROCESSED_DIR / 'customers_clean.csv', index=False, encoding='utf-8-sig')
products_clean.to_csv(PROCESSED_DIR / 'products_clean.csv', index=False, encoding='utf-8-sig')
orders_clean.to_csv(PROCESSED_DIR / 'orders_clean.csv', index=False, encoding='utf-8-sig')
order_items_clean.to_csv(PROCESSED_DIR / 'order_items_clean.csv', index=False, encoding='utf-8-sig')

list(PROCESSED_DIR.glob('*_clean.csv'))


[WindowsPath('c:/dev/llm-data-analysis-study/chapter05/data/processed/customers_clean.csv'),
 WindowsPath('c:/dev/llm-data-analysis-study/chapter05/data/processed/orders_clean.csv'),
 WindowsPath('c:/dev/llm-data-analysis-study/chapter05/data/processed/order_items_clean.csv'),
 WindowsPath('c:/dev/llm-data-analysis-study/chapter05/data/processed/products_clean.csv')]

## 19. 전처리 요약 보고서 저장하기

전처리 기준과 결과를 Markdown 파일로 남겨 두면 이후 보고서 작성이 쉬워집니다.


In [62]:
summary_text = f'''# Chapter 5 데이터 전처리 요약

## 전처리 결과 파일

- customers_clean.csv
- products_clean.csv
- orders_clean.csv
- order_items_clean.csv

## 전처리 전후 데이터 크기

```text
{comparison.to_string(index=False)}
```

## 파일 간 관계 점검 결과

```text
{relationship_checks.to_string(index=False)}
```

## 주요 처리 내용

- 원본 데이터를 직접 수정하지 않고 복사본을 사용함
- 문자열 컬럼 앞뒤 공백 제거
- 고객 나이 결측치는 중앙값으로 대체
- 고객 도시 결측치는 Unknown으로 처리
- 주문 상태값 표기 통일
- 날짜 컬럼 변환 및 주문 월/요일 파생 컬럼 생성
- 가격, 수량, 단가를 숫자형으로 변환
- 0 이하 가격, 수량, 단가 확인 및 처리
- line_total 파생 컬럼 생성
- 파일 간 키 관계 재확인

## 주의 사항

이 전처리 기준은 실습용 예시입니다. 실제 업무에서는 결측치와 이상값을 삭제하거나 대체하기 전에 원본 시스템, 수집 과정, 업무 담당자 확인이 필요합니다.
'''

report_path = REPORT_DIR / 'ch05_preprocessing_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('요약 보고서 저장 완료:', report_path)


요약 보고서 저장 완료: c:\dev\llm-data-analysis-study\chapter05\reports\ch05_preprocessing_summary.md


## 20. 전처리 과정을 함수로 정리하기

전처리 코드를 한 번만 실행하고 끝내면 재사용하기 어렵습니다. 같은 데이터를 다시 받거나 처리 기준을 조금 바꾸어야 할 때는 함수로 정리된 코드가 훨씬 유용합니다.


In [63]:
def preprocess_customers(df):
    result = strip_string_columns(df)

    if 'age' in result.columns:
        result['age'] = pd.to_numeric(result['age'], errors='coerce')
        result['age'] = result['age'].fillna(result['age'].median())

    if 'city' in result.columns:
        result['city'] = result['city'].fillna('Unknown')

    if 'signup_date' in result.columns:
        result['signup_date'] = pd.to_datetime(result['signup_date'], errors='coerce')

    return result.drop_duplicates()


def preprocess_products(df):
    result = strip_string_columns(df)

    if 'price' in result.columns:
        result['price'] = to_number(result['price'])
        result = result[result['price'] > 0]

    return result.drop_duplicates()


def preprocess_orders(df):
    result = strip_string_columns(df)

    if 'order_status' in result.columns:
        result['order_status'] = result['order_status'].replace(status_map)

    if 'order_date' in result.columns:
        result['order_date'] = pd.to_datetime(result['order_date'], errors='coerce')
        result['order_month'] = result['order_date'].dt.to_period('M').astype(str)
        result['order_dayofweek'] = result['order_date'].dt.day_name()

    return result.drop_duplicates()


def preprocess_order_items(df):
    result = strip_string_columns(df)

    if 'quantity' in result.columns:
        result['quantity'] = to_number(result['quantity'])
        result = result[result['quantity'] > 0]

    if 'unit_price' in result.columns:
        result['unit_price'] = to_number(result['unit_price'])
        result = result[result['unit_price'] > 0]

    if {'quantity', 'unit_price'}.issubset(result.columns):
        result['line_total'] = result['quantity'] * result['unit_price']

    return result.drop_duplicates()


In [64]:
customers_clean_func = preprocess_customers(customers)
products_clean_func = preprocess_products(products)
orders_clean_func = preprocess_orders(orders)
order_items_clean_func = preprocess_order_items(order_items)

print(customers_clean_func.shape)
print(products_clean_func.shape)
print(orders_clean_func.shape)
print(order_items_clean_func.shape)


(150, 6)
(100, 4)
(300, 7)
(764, 6)


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_1904\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_1904\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pan

## 21. 소스 모듈 사용하기

위에서 직접 작성한 전처리 함수는 `src/preprocessing.py`에도 정리되어 있습니다. 반복 작업이나 실제 프로젝트에서는 노트북 안에만 코드를 두기보다 소스 모듈로 분리하는 것이 좋습니다.


In [65]:
from src.data_loader import load_sales_data
from src.preprocessing import (
    build_preprocessing_report,
    compare_shapes,
    duplicate_summary,
    preprocess_sales_data,
    save_processed_data,
    validate_relationships,
)

module_raw_data = load_sales_data(RAW_DIR)
module_processed_data = preprocess_sales_data(module_raw_data)
module_relationship_checks = validate_relationships(module_processed_data)
module_comparison = compare_shapes(module_raw_data, module_processed_data)

module_comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,150,6,150,6
1,products,100,4,100,4
2,orders,300,5,300,7
3,order_items,764,5,764,6


In [71]:
duplicate_summary(
    module_processed_data,
    key_columns={
        'customers': 'customer_id',
        'products': 'product_id',
        'orders': 'order_id',
        'order_items': 'order_item_id',
    },
)


,dataset,full_row_duplicates,primary_key,key_duplicates
0,customers,0,customer_id,0
1,products,0,product_id,0
2,orders,0,order_id,0
3,order_items,0,order_item_id,0


In [72]:
module_relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,0
1,order_items.order_id exists in orders.order_id,0
2,order_items.product_id exists in products.prod...,0


## 22. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 전처리 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/preprocess_data.py
```

이 스크립트는 전처리 결과 CSV와 요약 보고서를 자동으로 저장합니다.


## 23. LLM과 함께 전처리 코드를 검토하기

LLM에게 전처리 코드를 요청할 때는 실제 고객명, 이메일, 주문 내역을 그대로 입력하지 않는 것이 좋습니다. 컬럼명, 데이터 타입, 결측치 개수, 중복 여부, 처리 목적처럼 구조화된 정보만 제공하세요.

```text
다음 전처리 코드가 안전한지 검토해 주세요.

customers["age"] = customers["age"].fillna(customers["age"].mean())
customers = customers.drop_duplicates()
orders["order_date"] = pd.to_datetime(orders["order_date"])
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

검토 기준:
- 원본 데이터를 직접 수정하는 문제가 있는지
- 결측치 처리 방식이 적절한지
- 날짜 변환 실패를 확인하는지
- 중복 제거 기준이 충분한지
- line_total 계산 전에 숫자형 변환이 필요한지
- 더 안전한 코드로 어떻게 수정할 수 있는지

검토 결과
- customers, orders, order_items에 바로 값을 대입하고 있어, 이 변수들이 원본 DataFrame이라면 메모리상의 원본이 변경된다. 원본을 보존하려면 처리 전에 복사본을 만들어 작업하는 것이 안전하다. 파일 자체를 덮어쓰는지는 저장 코드까지 확인해야 한다.
- age 컬럼이 실제로 존재하고 숫자형인지 먼저 확인해야 한다. 결측치를 평균으로 채우면 실제 나이 분포와 다른 값이 들어갈 수 있고, 평균보다 중앙값이나 별도 결측 표시가 적절한 경우도 있다. 
- pd.to_datetime()은 변환할 수 없는 값이 있으면 기본적으로 오류를 발생시킨다. 실행이 중단될 수 있고 실패 건수를 따로 확인하기 어렵다. errors='coerce'를 사용해 변환 실패를 NaT로 만든 다음, 변환 전후 실패 건수를 확인하는 편이 안전하다.
- drop_duplicates()는 모든 컬럼 값이 같은 행을 기준으로 삭제한다. 이 방식만으로는 order_id 같은 주요 키의 중복이나 업무상 중복을 모두 판단할 수 없다. 삭제 전에 중복 행과 키 중복을 확인하고 기준을 정해야 한다. 
- quantity와 unit_price가 숫자형이라는 보장이 없으므로, 계산 전에 숫자형 변환 여부와 변환 실패 건수를 확인해야 한다.
- 원본 DataFrame의 복사본에서 작업하고, 컬럼 존재 여부와 자료형을 확인한 뒤 변환해야 한다. 결측치 대체와 중복 제거는 확인된 문제와 기준이 있을 때만 적용하고, 날짜/숫자 변환 실패 건수와 처리 전후 행 수를 기록하는 방식이 적절하다.
```


## 24. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
2. `customers_clean`에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.
3. `orders_clean`에서 월별 주문 건수를 계산하세요.
4. `order_items_clean`에서 `line_total`이 큰 순서대로 상위 10개를 확인하세요.
5. 전처리 전후 행 수가 줄어든 데이터셋이 있는지 설명해 보세요.
6. LLM에게 결측치 처리 기준을 검토해 달라는 프롬프트를 직접 작성해 보세요.


In [78]:
# 과제 1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
# hint: missing_summary(df).reset_index()를 활용해 보세요.
datasets = {
    "customers": customers_clean,
    "products": products_clean,
    "orders": orders_clean,
    "order_items": order_items_clean,
}

missing_tables = []

for name, df in datasets.items():
    summary = missing_summary(df).reset_index()
    summary.insert(0, "dataset", name)
    missing_tables.append(summary)

missing_table = pd.concat(missing_tables, ignore_index=True)
missing_table

,dataset,index,missing_count,missing_ratio
0,customers,customer_id,0,0.0
1,customers,name,0,0.0
2,customers,gender,0,0.0
3,customers,age,0,0.0
4,customers,city,0,0.0
5,customers,signup_date,0,0.0
6,products,product_id,0,0.0
7,products,product_name,0,0.0
8,products,category,0,0.0
9,products,price,0,0.0


In [75]:
# 과제 2. customers_clean에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.
age_outliers = customers_clean[
    (customers_clean["age"] < 18) |
    (customers_clean["age"] > 100)
]

print("이상값 개수:", len(age_outliers))
age_outliers

이상값 개수: 0


,customer_id,name,gender,age,city,signup_date


In [76]:
# 과제 3. orders_clean에서 월별 주문 건수를 계산하세요.
orders_clean["order_date"] = pd.to_datetime(
    orders_clean["order_date"],
    errors="coerce"
)

monthly_orders = (
    orders_clean
    .dropna(subset=["order_date"])
    .groupby(orders_clean["order_date"].dt.to_period("M"))
    .size()
    .rename("order_count")
)

monthly_orders

order_date
2025-09     8
2025-10    25
2025-11    23
2025-12    30
2026-01    33
2026-02    14
2026-03    38
2026-04    26
2026-05    28
2026-06    24
2026-07    23
2026-08    20
2026-09     8
Freq: M, Name: order_count, dtype: int64

In [77]:
# 과제 4. order_items_clean에서 line_total이 큰 순서대로 상위 10개를 확인하세요.
order_items_clean.nlargest(10, "line_total")

,order_item_id,order_id,product_id,quantity,unit_price,line_total
22,23,10,99,5,200000,1000000
85,86,39,99,5,200000,1000000
183,184,76,99,5,200000,1000000
78,79,35,70,5,198000,990000
493,494,190,70,5,198000,990000
45,46,21,58,5,197000,985000
59,60,25,58,5,197000,985000
113,114,51,58,5,197000,985000
395,396,152,58,5,197000,985000
569,570,221,58,5,197000,985000


## 25. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 원본 데이터와 전처리 데이터 분리
- 결측치 개수와 비율 확인
- 나이 결측치 중앙값 대체, 도시 결측치 Unknown 처리
- 중복 행과 주요 ID 중복 점검
- 문자열 앞뒤 공백 제거와 상태값 표기 통일
- 날짜형 변환과 주문 월/요일 파생 컬럼 생성
- 숫자형 변환과 변환 실패 건수 확인
- 이상값 후보 확인과 실습용 처리 기준 적용
- 주문 상세 금액 `line_total` 생성
- 파일 간 키 관계 재확인
- 전처리 결과 CSV와 요약 보고서 저장
- 전처리 함수를 `src/preprocessing.py` 소스 모듈로 분리

다음 장에서는 전처리된 데이터를 바탕으로 탐색적 데이터 분석, 즉 EDA를 수행합니다.


# Chapter 05 제출 답안 양식. 분석을 믿을 수 있게 만드는 데이터 전처리

> 주 제출물은 실행 완료 Notebook `chapter05/chapter05.ipynb`입니다. 이 양식의 항목을 Notebook의 Markdown 셀로 추가해 작성합니다.

## 0. 제출 정보
- 이름: 황지원
- GitHub ID: Jiwon0712
- 작성일:2026-10-10
- 최종 제출 URL: 

## 1. 전처리 전 상태 기록
- 각 데이터 shape: customers (150,6), products (100,4), orders (300,5), order_items (764, 5)
- 결측치: customers, products, orders, order_items 전부 0건
- 전체 중복: customers, products, orders, order_items 전체 행 중복 0건
- 주요 ID 중복: 
customers customer_id 중복 수: 0
products product_id 중복 수: 0
orders order_id 중복 수: 0
order_items order_item_id 중복 수: 0
- 타입 문제 후보: 
customers. signup_date : str -> datetime 변환 필요
orders.order_date: str -> datetime 변환 필요
customer_id, product_id, order_id, order_item_idL int64. 합계/평균 대상에서 제외
customers.gender, customers.city, products.category, orders.payment_method, orders.order_status(범주형): str로 문제는 x, 고유값이 적어서 카테고리 변환 검토해보기.



![전처리 전 상태](images/step01_before.png)

### 결과 관찰
- 4개 테이블(customers 150행, products 100행, orders 300행, order_items 764행) 모두 결측치가 0건이고, 전체 행 중복과 주요 ID(customer_id, product_id, order_id, order_item_id) 중복도 0건이다.
- 값의 누락이나 중복 때문에 생기는 문제는 없고, 데이터 타입 문제만 확인하면 된다.
- 날짜 컬럼(customers.signup_date, orders.order_date)이 str 타입이라 datetime 변환이 필요하다.
- ID 컬럼(customer_id, product_id, order_id, order_item_id)이 int64라서 합계나 평균 같은 수치 집계에서 실수로 포함될 수 있다.
- 범주형 후보 컬럼(customers.gender, customers.city, products.category, orders.payment_method, orders.order_status)은 str 타입이다. 고유값 수가 적어 category 타입으로 바꾸기 좋다.

### 나의 해석과 판단
어떤 문제를 먼저 처리해야 하는지와 이유를 작성하세요.

- 가장 먼저 처리할 문제는 날짜 컬럼의 datetime 변환이다. 
- 이유: 날짜가 문자열이면  월별/요일별 주문 추이, 가입 시점 기준 코호트, 가입일과 주문일 간격 계산 같은 시계열 분석이 불가능하거나 정렬 오류가 생길 수 있다. 이후 분석 대부분이 이 변환에 의존한다. 또 결측치와 중복이 없으므로 데이터 정제 단계에서 제거하거나 보정할 대상이 없다. 

### 업무·분석적 의미
- 결측과 중복이 없어서 행 수가 줄지 않는다. 전처리 후에도 분석 모수가 그대로 유지되어 결과를 믿을 수 있다.
- 날짜를 datetime으로 바꾸면 매출 추이, 신규 가입 추이, 가입 후 첫 주문까지의 기간처럼 마케팅·운영에서 자주 쓰는 지표를 만들 수 있다.
- ID를 수치 집계에서 제외하면 의미 없는 통계값이 리포트에 섞이는 것을 막는다.
- 범주형을 정리해 두면 결제수단별, 지역별, 카테고리별, 주문상태별 비교를 일관되게 할 수 있다.

### 한계와 추가 확인 사항
- 결측치가 0건이라는 것은 NaN이 없다는 뜻일 뿐이다. "unknown", "-", 빈 문자열 같은 숨은 결측치가 있는지 확인해야 한다.
- 이상치는 아직 확인하지 않았다.
- 범주형 컬럼들의 표기 불일치를 고유값 목록으로 확인해야 한다.
- 테이블 간 참조 무결성을 확인해야 한다. 
- 날짜 범위가 타당한지 확인해야 한다.
- order_items의 금액이 products 가격이나 orders 합계와 일치하는지 아직 확인하지 않았다.

## 2. 문자열·타입·결측 처리
- 적용한 문자열 정리: 4개 테이블(customers, products, orders, order_items)의 문자열 컬럼 전체에 앞뒤 공백 제거를 적용했다. orders의 order_status 컬럼은 매핑 딕셔너리로 `complete/Complete/COMPLETED/완료`, `cancel/Cancel/CANCELLED/취소`, `refund/Refund/REFUNDED/환불`을 `completed/cancelled/refunded` 3개 값으로 통일했다. 숫자 컬럼은 변환 전에 천 단위 쉼표를 제거했다.
- 숫자 변환 대상과 실패 건수: 대상은 products의 price 컬럼, order_items의 quantity 컬럼, order_items의 unit_price 컬럼이고, 실패 건수는 각각 0건, 0건, 0건이다. 
- 날짜 변환 대상과 실패 건수: 대상은 customers의 signup_date 컬럼, orders의 order_date 컬럼이고, 실패 건수는 각각 0건, 0건이다. 
- 결측 처리 기준: 전처리 전 결측치가 0건이었고 변환 실패도 0건이라 새로 생긴 결측이 없다. 따라서 행 삭제나 값 대체는 하지 않았다. 

![타입과 결측 처리](images/step02_cleaning.png)

### 결과 관찰
- 4개 테이블의 문자열 컬럼에 앞뒤 공백 제거가 오류 없이 완료되었다. 실행 중 pandas 경고(`select_dtypes(include='object')` 관련)가 출력되었지만 결과에는 영향이 없다.
- city 상위 10개는 성남 21, 광주 17, 부산 16, 대구 15, 서울 15, 울산 14, 인천 14, 대전 14, 수원 13, 고양 11건이며, 표기가 갈라진 값은 보이지 않는다.
- order_status는 매핑 전후 모두 completed 184, cancelled 64, refunded 52건으로 같다. 합계는 300건으로 orders 행 수와 일치한다. 매핑으로 바뀐 값은 없었고, 처음부터 표준 표기였다.
- price, quantity, unit_price의 숫자 변환 실패는 모두 0건이다.
- signup_date와 order_date의 날짜 변환 실패는 모두 0건이다. 주문 기간은 2025-09-18부터 2026-09-17까지이고, 파생 컬럼 order_month와 order_dayofweek를 생성했다.

### 나의 해석과 판단
왜 해당 처리 방식을 선택했는지 작성하세요. 삭제/대체가 정보 손실을 만들 가능성도 설명하세요.

공백 제거, 표기 통일 매핑, `errors='coerce'` 변환 후 실패 건수 확인의 순서로 처리했고, 삭제와 대체는 하지 않았다.
- 공백 제거: "서울"과 "서울 "처럼 의미는 같은데 다른 값으로 집계되는 문제를 막는다. 값의 의미가 바뀌지 않으므로 정보 손실이 없다.
- 표기 통일 매핑: 상태값이 섞여 있으면 같은 상태가 따로 집계된다. 이번 데이터는 이미 표준 표기여서 바뀐 값이 없었지만, 이후 데이터가 들어와도 안전하도록 적용했다. 여러 표기가 하나로 합쳐지므로 원래 표기 구분은 사라지지만, 의미가 같은 값이라 분석에는 문제가 없다. 매핑에 없는 값은 그대로 남기 때문에 value_counts로 3개 값만 남았는지 확인했다.
- coerce 변환: 변환할 수 없는 값을 NaN/NaT로 바꾸면 원본 값이 사라져 정보 손실이 생길 수 있다. 그래서 변환 직후 실패 건수를 확인했고, 이번에는 모두 0건이라 손실이 없었다.
- 삭제/대체 미적용: 결측이 없어서 적용할 대상이 없다. 행을 삭제하면 모수가 줄고, 평균 등으로 대체하면 분포가 왜곡될 수 있으므로 필요할 때만 쓴다.

### 업무·분석적 의미
- 날짜가 datetime으로 바뀌어 월별/요일별 주문 분석을 바로 할 수 있다.
- price, quantity, unit_price가 숫자형이라 매출 같은 금액 집계를 계산할 수 있다.
- 취소 64건과 환불 52건은 전체 주문의 약 39%를 차지한다. 매출 집계에 completed만 포함할지 먼저 정의해야 지표가 왜곡되지 않는다.
- 문자열이 정리되어 지역별/상태별 집계와 테이블 조인의 신뢰도가 높아진다.

### 한계와 추가 확인 사항
- 변환에 성공했다는 것이 값이 타당하다는 뜻은 아니다. 음수 가격, 0 이하 수량, 비정상적으로 큰 금액 같은 이상치는 아직 확인하지 않았다.
- signup_date는 head 5건만 확인했다. 최솟값과 최댓값, 미래 날짜 여부, 가입일보다 앞선 주문일이 있는지 확인해야 한다.
- city는 상위 10개만 확인했다. 전체 고유값과 gender, products.category, payment_method의 고유값도 확인해야 한다.
- 공백 제거는 앞뒤만 처리한다. 문자열 중간의 공백이나 특수문자 같은 숨은 결측치는 남아 있을 수 있다.
- 쉼표 제거는 쉼표가 천 단위 구분자라는 가정에 기댄다. 다른 의미의 쉼표가 있으면 값이 잘못 합쳐질 수 있다.
- pandas 경고는 `include=['object', 'string']`으로 바꾸면 해소할 수 있다.

## 3. 중복·범주 표준화·이상값 후보
- 제거/보류한 중복: 제거한 중복은 없다. 전처리 전 확인 단계에서 4개 테이블의 전체 행 중복과 주요 ID 중복이 모두 0건.
- 표준화한 범주값: orders 테이블의 order_status 컬럼을 completed / cancelled / refunded 3개 값으로 통일했다. 이번 데이터는 이미 표준 표기여서 바뀐 값은 없었다.
- 허용값 밖 값: price ≤ 0은 0건, quantity ≤ 0은 0건, unit_price ≤ 0은 0건이다. 허용 기준은 세 컬럼 모두 0보다 큰 값이다.
- 이상값 후보: 허용값 밖 값은 없었다. `describe()` 기준으로 price는 5,000~200,000, unit_price는 5,000~200,000, quantity는 1~5 범위이고, 극단적으로 튀는 값은 보이지 않는다. 필터 적용 후 products (100, 4), order_items (764, 5)로 행 수가 그대로여서 제외된 행은 없다

![중복 범주 이상값](images/step03_rules.png)

### 결과 관찰
- price는 평균 110,040, 중앙값 112,000, 최솟값 5,000, 최댓값 200,000이다.
- unit_price는 평균 약 108,562, 중앙값 111,000, 최솟값 5,000, 최댓값 200,000이다. products의 price 컬럼과 범위가 같다.
- quantity는 평균 약 3.05, 중앙값 3, 최솟값 1, 최댓값 5이다. 값이 1~5 사이에 모여 있다.
- 세 컬럼 모두 0 이하 값이 0건이라 `> 0` 필터를 적용해도 행이 줄지 않았다.
- `describe()` 값으로 IQR 기준(Q1 − 1.5×IQR, Q3 + 1.5×IQR)을 계산해 보면 세 컬럼 모두 최솟값·최댓값이 경계 안에 있다. price 상한 경계는 약 303,875, unit_price는 약 310,125, quantity는 −1~7이다. 통계적 이상치 후보도 없다.

### 나의 해석과 판단
이상값을 자동 삭제하지 않고 후보로 검토해야 하는 이유를 작성하세요.

- 값이 통계적으로 크거나 작다고 해서 오류인 것은 아니다. 고가 상품이나 대량 주문처럼 실제로 발생한 정상 거래일 수 있다. 자동으로 지우면 실제 매출이 제외될 수 있다.
- 이상값이 입력 오류인지 실제 값인지는 데이터만으로 판단하기 어렵고 원본 시스템이나 담당자 확인이 필요하다. 먼저 후보로 남겨 두고 확인한 뒤 처리해야 한다.
- 행을 삭제하면 분석 모수가 줄고, 같은 주문의 다른 컬럼 정보까지 함께 사라진다. 삭제는 되돌리기 어려우므로 근거가 확인된 뒤에 하는 것이 안전하다.
- 따라서 이번 데이터에서는 0 이하 값(허용값 밖)과 IQR 기준 이상치 후보가 모두 없어서 삭제나 보정은 하지 않았다. `> 0` 필터는 실습용 기준으로 적용했고, 실제 업무라면 적용 전에 담당자 확인이 필요하다.

### 업무·분석적 의미
- 가격과 수량에 비정상 값이 없어 매출 집계 결과를 신뢰할 수 있다.
- 행이 줄지 않아 이후 분석의 모수가 그대로 유지된다.
- order_status 컬럼이 3개 값으로 정리되어 상태별 집계가 일관된다.
- 중복이 없어서 주문과 주문 상세의 건수가 부풀려지지 않는다.

### 한계와 추가 확인 사항
- 허용값 기준이 0 초과뿐이다. 수량이나 단가의 상한은 정의하지 않았다. quantity 1~5가 업무 규칙상 정상 범위인지는 별도로 확인해야 한다.
- 이상값 검사는 products의 price 컬럼, order_items의 quantity 컬럼, order_items의 unit_price 컬럼에만 했다. 다른 수치 컬럼이 있는지 추가 확인하지 못했다.
- order_items의 unit_price 컬럼이  products의 price 컬럼과 일치하는지, 같은 상품의 단가가 주문마다 다른지는 확인하지 않았다.
- quantity에 소수 같은 비정수 값이 있는지는 `describe()`만으로는 알 수 없다.
- 중복 확인은 문자열 정리 전 기준이다. 공백 제거와 표기 통일 이후에 중복이 새로 생겼는지 추가로 확인하는 것이 좋다. 
- gender, products의 category, payment_method 컬럼의 표기 표준화는 이미지에서 확인하지 못했다.
- IQR 계산은 `describe()` 값으로 직접 한 것이며, 노트북에서 별도 이상치 검사를 실행한 결과는 아니다.

## 4. PK/FK와 파생 컬럼 재검증
- 없는 `customer_id`: 0건 
- 없는 `order_id`: 0건
- 없는 `product_id`: 0건
- 생성한 파생 컬럼: `line_total` (= `quantity` × `unit_price`)를 order_items_clean에 생성했다. 주문 상세 한 줄의 금액이다.
- `line_total` 검증: 상위 5행에서 quantity × unit_price와 line_total이 모두 일치했다(3×102,000=306,000, 5×25,000=125,000, 3×142,000=426,000, 3×193,000=579,000, 4×189,000=756,000). 전처리 후 line_total 전체 합계는 255,610,000이다.

![관계와 파생 컬럼 검증](images/step04_validation.png)

### 결과 관찰
- 관계 검증 3건(orders/customers, order_items/orders, order_items/products)의 invalid_count가 모두 0이다. 전처리 후 기준으로 고아 행은 없다.
- line_total은 `quantity * unit_price`로 계산했고 상위 5행의 값이 수식과 일치한다.
- 전처리 후 line_total 합계는 255,610,000이다. 764행으로 나누면 행당 평균 약 334,568이다. 이는 앞 단계 `describe()`의 quantity 평균(약 3.05)과 unit_price 평균(약 108,562)을 곱한 값(약 331,000)과 비슷한 수준이다.


### 나의 해석과 판단
전처리 후 관계를 다시 검증해야 하는 이유를 작성하세요.
- 전처리는 값을 바꾸거나 행을 지우는 작업이다. 공백 제거, 타입 변환, `> 0` 필터 같은 처리가 키 값을 바꾸거나 부모 행을 없애면 조인이 끊길 수 있다. 처리 후 결과를 기준으로 확인해야 이후 분석이 안전하다.
- 관계가 깨져 있으면 조인 시 행이 사라지거나 NaN이 생겨 매출과 고객별 집계가 틀어진다. 파생 컬럼 line_total도 정리된 quantity, unit_price에 의존하므로 같이 검증해야 한다.
- 이번 결과에서는 고아 키가 0건이고 필터 전후 행 수가 같아 전처리로 관계가 손상되지 않았다고 볼 수 있다.

## 5. 전처리 전/후 비교

| 항목 | 처리 전 | 처리 후 | 변화 이유 |
|---|---|---|---|
| 행 수 | customers 150 / products 100 / orders 300 / order_items 764 | customers 150 / products 100 / orders 300 / order_items 764 | 결측, 중복, 0 이하 값이 없어 삭제한 행이 없다. `> 0` 필터를 적용했지만 제외된 행도 없다. |
| 열 수 | customers 6 / products 4 / orders 5 / order_items 5 | customers 6 / products 4 / orders 7 / order_items 6 | 파생 컬럼 추가: orders에 order_month, order_dayofweek(+2), order_items에 line_total(+1) |
| 결측 | 4개 테이블 모두 0건 | 0건 (변환 실패 0건 기준) | 원래 결측이 없었고 변환 실패(NaN/NaT)로 새로 생긴 결측도 없다. 삭제나 대체는 하지 않았다. |
| 중복 | 0건 (전체 행, 주요 ID) | 재확인 안 함 (행 수 변화 없음) | 제거한 중복이 없다. 공백 제거와 표기 통일 이후의 중복은 재확인하지 않았다. |
| 변환 실패 | 해당 없음 (변환 전) | 0건 (price, quantity, unit_price, signup_date, order_date 모두) | `errors='coerce'` 변환 후 실패 건수를 확인했고 모두 0건이다. |

![전처리 전후 비교](images/step05_before_after.png)

### 결과 관찰
- 4개 테이블 모두 처리 전후 행 수가 같다. 합계는 1,314행에서 1,314행이다.
- customers와 products는 열 수도 처리 전후 동일하다. 데이터 타입만 바뀌었다.
- orders는 5열에서 7열, order_items는 5열에서 6열로 늘었다. 증가한 열은 모두 파생 컬럼이다.
- order_status는 표기 통일 매핑 전후 건수가 같다(completed 184, cancelled 64, refunded 52).
- 변환 실패는 숫자 3개 컬럼과 날짜 2개 컬럼 모두 0건이다.

### 나의 해석과 판단
행 수나 값 분포 변화가 의도한 결과인지 판단한 근거를 작성하세요.

행 수가 그대로인 것은 의도한 결과라고 판단했다.
- 1단계에서 결측과 중복이 0건이었고, 3단계에서 0 이하 값도 0건이었다. 삭제 대상이 없으므로 행 수가 같아야 정상이다. 만약 줄었다면 어떤 규칙이 행을 지웠는지 추적해야 했다.
- 늘어난 열은 order_month, order_dayofweek, line_total이다. 모두 분석을 위해 의도적으로 만든 파생 컬럼이고, 원본 컬럼은 삭제하지 않았다.
- 전처리는 타입 변환(문자열→날짜/숫자)과 공백 제거가 중심이었다. 표기 통일 매핑으로도 건수가 바뀌지 않았다. 따라서 값 분포가 달라질 만한 처리는 없었다고 판단했다.

### 업무·분석적 의미
- 분석 모수가 처리 전과 같아서 처리 후 지표를 원본 데이터와 같은 기준으로 비교할 수 있다.
- 파생 컬럼 덕분에 월별/요일별 주문 분석과 주문 상세 금액 집계를 바로 진행할 수 있다.
- 이번 결과는 정리할 문제가 적은 데이터라는 뜻이다. 실제 업무 데이터에서는 행 수가 줄거나 결측이 생기는 경우가 많아서, 이 비교표로 처리 영향 범위를 기록해 두는 것이 중요하다.

### 한계와 추가 확인 사항
- 처리 후 결측과 중복은 비교 이미지에서 직접 확인한 값이 아니다. 결측은 변환 실패 0건과 행 수 불변으로 추정했고, 중복은 재확인하지 않았다. 전처리 후 `isna().sum()`과 `duplicated().sum()`을 다시 실행해 채우는 것이 안전하다.
- 행 수와 열 수만 비교했다. 컬럼별로 평균이나 고유값의 수 전후 비교는 하지 않았다.
- 처리 전 `describe()`가 없어서 price, quantity, unit_price의 전후 분포는 비교하지 못했다.
- 컬럼의 dtype 변경은 열 수에 드러나지 않으므로 전후 `dtypes`를 따로 비교하면 좋다.
- 처리 전 변환 실패 건수는 측정 대상이 아니어서 비워 두었다.

## 6. 재현 가능한 전처리 확인
- 생성된 clean CSV: `customers_clean.csv`, `products_clean.csv`, `orders_clean.csv`, `order_items_clean.csv` 4개를 `chapter05/data/processed/`에 저장했다. `index=False`, `encoding='utf-8-sig'`로 저장했다.
- 생성된 요약 보고서:`chapter05/reports/ch05_preprocessing_summary.md`
- `python scripts/preprocess_data.py` 재실행 결과: 성공
행 수: customers 150, products 100, orders 300, order_items 764 — 처리 전후 동일
관계 검증 오류: 모두 0건
전체 행 중복 및 기본 키 중복: 모두 0건
정제 CSV 4개와 요약 보고서 저장 완료  
- 재실행 후 달라진 점: 전처리 결과의 shape와 열 수는 Notebook에서 셀 단위로 처리한 결과와 같다(5단계 비교표와 일치). 달라진 점은 실행 방식이다. 처음에는 `scripts/preprocess_data.py`와 `src` 모듈이 없어 실행되지 않았고, 모듈과 스크립트를 새로 만들었다. 원본 폴더를 자동으로 찾지 못해(`chapter05/data/raw`에는 없고 저장소 루트의 `data/raw`에 있음) `--raw-dir`로 지정해야 했다. 실행하면서 clean CSV 4개와 요약 보고서를 새로 덮어썼다. 

![재실행 결과](images/step06_reproducible.png)

### 나의 해석과 판단
Notebook에서 한 번 성공하는 것과 스크립트로 재현 가능한 것의 차이를 작성하세요.
- Notebook에서 한 번 성공적으로 실행되었다고 해서 같은 작업을 언제든 다시 재현할 수 있는 것은 아니다.Notebook은 셀을 실행한 순서나 이전에 메모리에 남아 있는 변수에 영향을 받기 때문에, 다른 환경에서 처음부터 실행하면 오류가 날 수 있다. 반면 스크립트는 필요한 모듈과 입력 경로를 명시하고 정해진 순서로 실행하므로, 실행 조건을 갖추면 같은 전처리 결과를 반복해서 만들 수 있다.
- 이번에는 처음에 src 모듈을 찾지 못하고 원본 데이터 경로도 맞지 않아 스크립트 실행에 어려움이 있었다. 모듈과 스크립트를 정리하고 --raw-dir로 실제 데이터 폴더를 지정한 뒤에는 전처리가 완료되어 정제 CSV 4개와 요약 보고서가 저장되었다. 이 과정에서 재현성은 스크립트 파일만 만드는 것으로 확보되는 게 아니라, 모듈 구성과 데이터 경로 같은 실행 조건도 함께 정리해야 한다는 점을 알 수 있었다. 앞으로는 실행 방법과 필요한 경로를 기록하고, 결과 파일이 덮어써지는 점도 고려해 스크립트를 관리해야겠다고 판단했다.

## 7. Chapter 05 최종 판단
### 내가 정한 전처리 원칙 3가지
1. 원본 데이터는 보존하고, 변경 이유와 결과를 기록한다.
2. 데이터의 의미를 확인한 뒤 필요한 변환만 한다.
3. 처리 후에도 데이터의 수와 관계가 유지되는지 검증한다.

### 가장 위험하다고 생각한 자동 처리 1가지와 이유
이상값을 기준만으로 자동 삭제하는 것이 가장 위험한 것 같다. 값이 크거나 작아도 정상적인 값일 수 있는데, 이를 오류로 보고 삭제하면 실제 매출과 분석 대상이 줄어들 수 있다. 이상값 후보를 먼저 확인 후, 업무 규칙이나 원본 자료로 오류가 확인된 경우에만 삭제하거나 수정한다. 

### 다음 EDA에서 특히 주의할 데이터 특성
- 주문 300건 중 취소 64건, 환불 52건으로 두 상태가 총 116건, 약 38.7%를 차지한다. 따라서 매출을 계산할 때 취소/환불 주문을 포함할지 기준을 먼저 정해야 한다.
- 주문 기간은 2025년 9월 18일부터 2026년 9월 17일까지다. 월별/요일별 추이를 살펴볼 수 있지만, 약 1년치 자료만으로 장기적인 계절성을 단정하지 않도록 주의해야 한다.
- order_items의 unit_price와 products의 price가 상품별로 일치하는지 확인할 필요가 있다. 가격이 다르다면 주문 당시 가격인지, 입력 오류인지 추가로 판단해야 한다.

### 현재 전처리의 한계
현재는 결측치·중복·기본적인 값 범위와 테이블 관계를 확인했지만, 모든 값이 업무상 타당한지까지 검증한 것은 아니다. 예를 들어 가입일과 주문일의 논리적 관계, 주문 상세 단가와 상품 가격의 일치 여부, 수량의 업무상 허용 범위는 추가 확인이 필요하다. 또한 재실행 과정에서 입력 데이터 경로를 직접 지정해야 했으므로, 다른 환경에서도 같은 방식으로 실행하려면 경로와 필요한 실행 조건을 함께 기록해야 한다.

## 최종 제출 체크
- [x] raw 원본을 덮어쓰지 않았습니다.
- [x] 처리 전/후 비교를 남겼습니다.
- [x] 처리 기준과 이유를 자신의 말로 작성했습니다.
- [x] clean CSV와 재실행 결과를 확인했습니다.
- [x] 개인정보/Secret이 없습니다.
- [x] `chapter05/chapter05.ipynb`가 GitHub에서 정상 표시됩니다.
- [x] 최종 Notebook 파일 URL을 제출합니다.